# TVA Grid Sensor Placement: starter notebook

A PMU (phasor measurement unit) at a substation measures that substation and every substation one power line away. PMUs are expensive. Your job: find the smallest set of substations that lets TVA's operators see every substation.

In this notebook you will:
1. Draw the IEEE 14-bus test grid and place sensors by hand.
2. Build two classical baselines: a greedy rule and an exact integer program.
3. Write the cost function that QAOA will minimize, with one qubit per substation.
4. Run QAOA on a simulator, first on a 6-substation piece, then on all 14.
5. Run the 6-substation piece on IonQ through qBraid.

**Your 3 TODOs:** the greedy rule, the cost function, and the QAOA cost layer. Everything else is provided and working. A TODO raises `NotImplementedError` until you fill it in.

Words we use: a **bus** is a substation, a **line** is a power line, and a bus is **seen** when at least one sensor sees it.

In [ ]:
# qBraid Lab: install Qiskit Aer if this environment doesn't have it yet (one time, ~20 s)
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer

import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from scipy.optimize import milp, LinearConstraint, Bounds, minimize

# check.py sits next to this notebook: it loads the grids and checks placements
from check import load_grid, seen_buses, check_placement, brute_force_best, bits_to_buses

sim = AerSimulator()

## 1. Load and draw the grid

`grid["buses"]` lists the bus numbers, `grid["lines"]` lists the power lines, and `grid["neighbors"][b]` is the set of buses one line away from bus `b`. Later, qubit `i` stands for bus `grid["buses"][i]`.

`piece` is a 6-substation piece of the same grid, small enough for real quantum hardware.

In [ ]:
grid = load_grid("data/ieee14.json")
piece = load_grid("data/ieee14_piece6.json")
print(grid["name"], "|", len(grid["buses"]), "buses,", len(grid["lines"]), "lines")
print(piece["name"], "| buses", piece["buses"])


def layout(g):
    # networkx is only used to lay out the drawing; without it, buses go on a circle
    try:
        import networkx as nx
        return nx.kamada_kawai_layout(nx.Graph(g["lines"]))
    except ImportError:
        angle = np.linspace(0, 2 * np.pi, len(g["buses"]), endpoint=False)
        return {b: (np.cos(a), np.sin(a)) for b, a in zip(g["buses"], angle)}


def draw_grid(g, chosen=(), title="", pos=None):
    pos = pos or layout(g)
    seen = seen_buses(g, chosen)
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for a, b in g["lines"]:
        ax.plot(*zip(pos[a], pos[b]), color="#b9b8b3", lw=2, zorder=1)
    for b in g["buses"]:
        face = "#2a78d6" if b in chosen else ("#cde2fb" if b in seen else "white")
        ring = "#2a78d6" if b in seen else "#d03b3b"
        ax.scatter(*pos[b], s=450, c=face, edgecolors=ring, linewidths=2, zorder=2)
        ax.annotate(str(b), pos[b], ha="center", va="center", fontsize=9, zorder=3,
                    color="white" if b in chosen else "black")
    for label, face, ring in [("sensor", "#2a78d6", "#2a78d6"), ("seen", "#cde2fb", "#2a78d6"), ("unseen", "white", "#d03b3b")]:
        ax.scatter([], [], s=120, c=face, edgecolors=ring, linewidths=2, label=label)
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
    ax.set_title(title or g["name"])
    ax.margins(0.08)
    ax.axis("off")
    plt.show()


pos14 = layout(grid)              # one layout, so the piece sits where it is in the full grid
draw_grid(grid, pos=pos14)
draw_grid(piece, pos=pos14)

## Milestone 1: place sensors by hand

Look at the drawing of the 14-bus grid. Put bus numbers in `my_guess`, run the cell, and repeat until every bus is seen with as few sensors as you can manage.

In [ ]:
my_guess = [4, 6]          # edit this list of bus numbers
check_placement(grid, my_guess)
draw_grid(grid, my_guess, "My guess", pos=pos14)

## Milestone 2: classical baselines

- **Greedy (your TODO):** keep adding the sensor that sees the most still-unseen buses. Fast, but not always the fewest.
- **Integer program (provided):** an ILP (integer linear program) gives each bus a 0/1 variable $x_b$ and minimizes $\sum_b x_b$ so that every bus has at least one sensor among itself and its neighbors. `scipy.optimize.milp` solves it exactly.

Done when you have the optimal count and its runtime.

In [ ]:
def greedy_placement(g):
    # Add sensors one at a time, always at the bus that sees the most unseen buses.
    chosen = []
    unseen = set(g["buses"])
    # TODO (Milestone 2): while some bus is unseen, pick the bus b whose group
    # {b} | g["neighbors"][b] holds the most unseen buses, add b to `chosen`,
    # and remove that group from `unseen`.
    # Hint: b = max(g["buses"], key=lambda b: len(({b} | g["neighbors"][b]) & unseen))
    raise NotImplementedError("Write the greedy rule")
    return chosen

In [ ]:
def ilp_placement(g, twice=False):
    # Exact minimum: one 0/1 variable per bus, and for every bus
    # (sensors on it or its neighbors) >= 1, or >= 2 with twice=True (N-1 stretch)
    buses = g["buses"]
    A = np.array([[1 if c == b or c in g["neighbors"][b] else 0 for c in buses] for b in buses])
    res = milp(c=np.ones(len(buses)), constraints=LinearConstraint(A, lb=2 if twice else 1),
               integrality=np.ones(len(buses)), bounds=Bounds(0, 1))
    return [b for b, x in zip(buses, res.x) if x > 0.5]


for g in [piece, grid]:
    for name, method in [("greedy", greedy_placement), ("ILP", ilp_placement)]:
        try:
            t0 = time.perf_counter()
            chosen = method(g)
            ms = 1000 * (time.perf_counter() - t0)
        except NotImplementedError:
            print(f"{name}: TODO\n")
            continue
        print(f"{name}: {chosen} in {ms:.1f} ms")
        check_placement(g, chosen)
        print()

draw_grid(grid, ilp_placement(grid), "ILP placement", pos=pos14)

## Milestone 3: the cost function

Give each bus one qubit: $x_b = 1$ means "put a sensor at bus $b$". QAOA will look for the bitstring with the lowest cost

$$C(x) = \underbrace{\sum_b x_b}_{\text{sensors}} \; + \; P \sum_b \underbrace{\prod_{j \in N[b]} (1 - x_j)}_{1 \text{ if bus } b \text{ is unseen}}$$

where $N[b]$ is bus $b$ plus its neighbors (its "group"). The product is 1 only when every bus in the group has no sensor, which is exactly when bus $b$ is unseen.

**Pick $P > 1$**, so an unseen bus costs more than one sensor. Then leaving a bus unseen always costs more than adding a sensor that sees it, so the cheapest bitstring is a valid placement with the fewest sensors. We use $P = 2$.

Why this form: it needs exactly one qubit per bus. The textbook QUBO (a cost with only pairs of 0/1 variables) turns each "at least one sensor" rule into an equation with extra slack qubits, about three times as many qubits for IEEE 14 (46 instead of 14).

**Bit order:** in this notebook a bitstring like `"010010"` lists qubit 0 first, so character `i` is bus `g["buses"][i]`. Qiskit prints qubit 0 *last*; the provided helpers handle the flip.

In [ ]:
def bus_groups(g):
    # groups[i] = the qubits of bus i and its neighbors (qubit i is bus g["buses"][i])
    index = {b: i for i, b in enumerate(g["buses"])}
    return [sorted(index[c] for c in {b} | g["neighbors"][b]) for b in g["buses"]]


P = 2.0   # penalty per unseen bus, must be > 1

def placement_cost(bits, g, P=P):
    # C(x) = number of sensors + P * number of unseen buses. bits[i] is "0" or "1" for qubit i.
    x = [int(c) for c in bits]
    groups = bus_groups(g)
    # TODO (Milestone 3): count the sensors, count the unseen buses (a bus is unseen
    # when every qubit in its group is 0), and return sensors + P * unseen.
    # Hint: unseen = sum(1 for grp in groups if all(x[j] == 0 for j in grp))
    raise NotImplementedError("Write the cost function")

In [ ]:
def cost_table(g, P=P):
    # Cost of every bitstring, in the order of Qiskit's statevector: bit i of index z is qubit i.
    # format(z, "06b") puts qubit 0 last (Qiskit style), so [::-1] puts it first (our style).
    n = len(g["buses"])
    return np.array([placement_cost(format(z, f"0{n}b")[::-1], g, P) for z in range(2 ** n)])


costs6 = cost_table(piece)
z = int(np.argmin(costs6))
cheapest = bits_to_buses(piece, format(z, "06b")[::-1])
print("Cheapest bitstring by your cost function ->", cheapest, "| cost", costs6[z])
check_placement(piece, cheapest)
print("ILP answer:", ilp_placement(piece))

## Milestone 4: QAOA

QAOA starts with every qubit in $|+\rangle$ (all placements at once) and repeats two layers `layers` times:
- **cost layer** (your TODO): multiply each bitstring $x$ by the phase $e^{-i\gamma C(x)}$
- **mixer layer** (provided): `rx(2*beta)` on every qubit

A classical optimizer tunes the angles $\gamma, \beta$ (provided). The cost layer has two kinds of terms:
1. **Sensor cost** $x_b$: a phase $e^{-i\gamma}$ when qubit $i$ is 1. That is `qc.p(-gamma, i)`.
2. **Unseen penalty** $P \prod_{j \in N[b]} (1 - x_j)$: a phase $e^{-i\gamma P}$ only when every qubit in the group is 0. A multi-controlled phase `mcp` fires only when all its qubits are 1, so flip the group with X gates, apply `mcp`, and flip back.

This keeps the circuit at one qubit per bus with no extra qubits.

In [ ]:
def cost_layer(qc, g, gamma, P=P):
    # Apply exp(-i * gamma * C(x)) to every bitstring x.
    groups = bus_groups(g)
    # TODO (Milestone 4): add both kinds of phase.
    # 1) Sensor cost: for every qubit i, qc.p(-gamma, i)
    # 2) Unseen penalty: for every group grp, qc.x(grp), then
    #    qc.mcp(-gamma * P, grp[:-1], grp[-1]), then qc.x(grp) again.
    raise NotImplementedError("Write the cost layer")


def check_cost_layer(g, gamma=0.37):
    # Compare your circuit's phases with exp(-i * gamma * C) from placement_cost.
    n = len(g["buses"])
    qc = QuantumCircuit(n)
    qc.h(range(n))
    cost_layer(qc, g, gamma)
    qc.save_statevector()
    state = np.asarray(sim.run(transpile(qc, sim)).result().get_statevector())
    ratio = state / (np.exp(-1j * gamma * cost_table(g)) / np.sqrt(2 ** n))
    ok = np.allclose(ratio, ratio[0], atol=1e-6)   # equal up to one global phase
    print(g["name"], "| cost layer matches your cost function:", ok)
    return ok


check_cost_layer(piece)

### The QAOA loop (provided)

`run_qaoa` tunes the angles on the exact statevector, which is fast for 14 qubits. Two knobs to try:
- `layers`: more layers usually help, but make a longer circuit.
- `alpha`: `alpha=1` is textbook QAOA (minimize the average cost). `alpha=0.25` minimizes the average of the cheapest 25% of outcomes (called CVaR), which rewards QAOA for its best answers and usually finds the optimum more often.

In [ ]:
def qaoa_circuit(g, params, P=P, measure=False):
    # params = [gamma_1 .. gamma_L, beta_1 .. beta_L]
    L = len(params) // 2
    n = len(g["buses"])
    qc = QuantumCircuit(n, n) if measure else QuantumCircuit(n)
    qc.h(range(n))
    for gamma, beta in zip(params[:L], params[L:]):
        cost_layer(qc, g, gamma, P)
        qc.rx(2 * beta, range(n))          # mixer
    if measure:
        qc.measure(range(n), range(n))
    else:
        qc.save_statevector()
    return qc


def qaoa_probs(g, params, P=P):
    # Exact chance of every bitstring (index z: bit i of z is qubit i)
    state = sim.run(transpile(qaoa_circuit(g, params, P), sim)).result().get_statevector()
    return np.abs(np.asarray(state)) ** 2


def score(probs, costs, alpha):
    # Average cost over the cheapest `alpha` share of outcomes (alpha=1: plain average)
    order = np.argsort(costs)
    pr, c = probs[order], costs[order]
    take = np.clip(alpha - (np.cumsum(pr) - pr), 0, pr)
    return take @ c / alpha


def run_qaoa(g, layers=1, P=P, alpha=0.25, maxiter=100):
    costs = cost_table(g, P)
    objective = lambda params: score(qaoa_probs(g, params, P), costs, alpha)
    k = (np.arange(layers) + 0.5) / layers
    # a few "ramp" starts (gamma grows, beta shrinks, like a slow anneal); keep the best
    starts = [np.concatenate([k * a, (1 - k) * b]) for a, b in [(0.2, 0.2), (0.3, 0.6), (0.6, 0.3), (0.4, 0.4)]]
    x0 = min(starts, key=objective)
    result = minimize(objective, x0, method="COBYLA", options={"maxiter": maxiter, "rhobeg": 0.2})
    return result.x, qaoa_probs(g, result.x, P)


def outcome_shares(g, probs):
    # Split the probability into: optimal, valid with extra sensors, leaves a bus unseen
    n = len(g["buses"])
    z = np.arange(2 ** n)
    sensors = sum((z >> i) & 1 for i in range(n))
    unseen = sum((z & sum(1 << j for j in grp)) == 0 for grp in bus_groups(g))
    best, _ = brute_force_best(g)
    valid = unseen == 0
    return {"optimal": probs[valid & (sensors == best)].sum(),
            "valid, extra sensors": probs[valid & (sensors > best)].sum(),
            "leaves a bus unseen": probs[~valid].sum()}


def plot_outcomes(rows, title):
    colors = {"optimal": "#0ca30c", "valid, extra sensors": "#fab219", "leaves a bus unseen": "#d03b3b"}
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(rows) + 1.2))
    labels = list(rows)[::-1]
    left = np.zeros(len(labels))
    for kind, color in colors.items():
        vals = np.array([rows[r][kind] for r in labels])
        ax.barh(labels, vals, left=left, color=color, edgecolor="white", linewidth=2, label=kind)
        for y, (l, v) in enumerate(zip(left, vals)):
            if v > 0.06:
                ax.text(l + v / 2, y, f"{v:.0%}", ha="center", va="center", fontsize=9)
        left += vals
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel("share of shots")
    ax.set_title(title, pad=26)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()
    for r, s in rows.items():
        print(f"{r:28s}" + "  ".join(f"{k}: {v:6.2%}" for k, v in s.items()))

In [ ]:
layers6 = 1     # keep 1 for hardware: a short circuit
t0 = time.perf_counter()
params6, probs6 = run_qaoa(piece, layers=layers6)
print(f"6-piece: {time.perf_counter() - t0:.1f} s, angles {np.round(params6, 3)}")
for z in np.argsort(probs6)[::-1][:3]:
    print(f"  {probs6[z]:.1%}  sensors at {bits_to_buses(piece, format(z, '06b')[::-1])}")

In [ ]:
layers14 = 3
t0 = time.perf_counter()
params14, probs14 = run_qaoa(grid, layers=layers14)
print(f"14 buses: {time.perf_counter() - t0:.1f} s, angles {np.round(params14, 3)}")
for z in np.argsort(probs14)[::-1][:3]:
    print(f"  {probs14[z]:.1%}  sensors at {bits_to_buses(grid, format(z, '014b')[::-1])}")

In [ ]:
n6, n14 = len(piece["buses"]), len(grid["buses"])
plot_outcomes({
    f"6-piece, QAOA layers={layers6}": outcome_shares(piece, probs6),
    "6-piece, random guess": outcome_shares(piece, np.full(2 ** n6, 2.0 ** -n6)),
    f"14 buses, QAOA layers={layers14}": outcome_shares(grid, probs14),
    "14 buses, random guess": outcome_shares(grid, np.full(2 ** n14, 2.0 ** -n14)),
}, "What one QAOA shot gives you")

## Milestone 5: run the 6-substation piece on IonQ

1. Run the cell below as is. It sends your tuned 6-piece circuit to IonQ's **free** simulator with a realistic noise model (`aria-1`). Compare settings there (for example `layers6 = 1` vs `2`, or `alpha`) before you spend credits.
2. When you're happy, swap in the IonQ Forte QPU lines and run once at **200 shots or fewer**. Hardware runs close **Saturday 9 PM**.
3. Copy the job ID into your submission. If the queue is stuck, the job ID plus your noisy-simulator result counts in full.

In [ ]:
# Your tuned 6-piece circuit, written in basic gates so the two-qubit count is the real one
qc = transpile(qaoa_circuit(piece, params6, measure=True), basis_gates=["rz", "sx", "x", "cx"])
print("Qubits:", qc.num_qubits, "| two-qubit (CX) gates:", qc.count_ops()["cx"], "(IonQ advises a few dozen at most)")
# qBraid may warn that measurements are ignored: IonQ always measures every qubit, so that's fine

# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Practice for free: IonQ's simulator with a realistic noise model
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 2) Your real hardware run (uses credits). Swap in the QPU when you're ready:
# device = provider.get_device("ionq:ionq:qpu:forte-1")
# job = device.run(qc, shots=200)

print("Job ID:", job.id)          # copy this into your submission
counts = job.result().data.get_counts()
print(counts)

In [ ]:
probs_hw = np.zeros(2 ** n6)
for key, c in counts.items():
    probs_hw[int(key.replace(" ", ""), 2)] += c   # int(key, 2) reads Qiskit's order: bit i is qubit i
probs_hw /= probs_hw.sum()

plot_outcomes({
    "Ideal simulator": outcome_shares(piece, probs6),
    f"IonQ run ({sum(counts.values())} shots)": outcome_shares(piece, probs_hw),
    "Random guess": outcome_shares(piece, np.full(2 ** n6, 2.0 ** -n6)),
}, "6-piece: IonQ vs ideal simulator")
top = max(counts, key=counts.get)
print("Most common IonQ answer: sensors at", bits_to_buses(piece, top.replace(" ", "")[::-1]))

## Stretch ideas

- **Zero-injection buses:** `check_placement(grid, chosen, zib=True)` applies the rule (IEEE 14 bus 7 has no load and no generator). How many sensors can you save?
- **N-1:** `check_placement(grid, chosen, n1=True)` and `ilp_placement(grid, twice=True)`. For QAOA, add a penalty for buses seen only once.
- **Critical buses:** give some buses a bigger penalty in `placement_cost` and `cost_layer` (for example, buses with big loads in `grid["load_mw"]`).
- **30 buses:** load `data/ieee30.json`. The exact cost table no longer fits in memory, so switch to sampled shots and qBraid's GPU simulator.

## What to submit

| Milestone | Submit |
|---|---|
| 1. Understand it | your hand placement and one sentence on what a PMU sees |
| 2. Classical baseline | greedy and ILP placements, their sensor counts and runtimes |
| 3. Cost function | your `placement_cost`, and the check showing its cheapest answer matches the ILP on the 6-piece |
| 4. QAOA | the outcome chart for the 6-piece and 14 buses, with your `layers`, `P` and `alpha` |
| 5. IonQ run | the job ID(s), the CX count, and the IonQ vs simulator chart |
| 6. Tell the story | up to 5 slides: what the sensors cost, what an operator gains, where quantum honestly stands |

Also hand in a 5-line AI log (what you asked, what you kept, what you fixed).